In [2]:
import pandas as pd
import numpy as np
from google.colab import files
import uuid

In [3]:
# Step 1: Upload and load dataset
uploaded = files.upload()
filename = list(uploaded.keys())[0]  # Get the first uploaded file
df = pd.read_csv(filename)

Saving Japan stock prediction value.csv to Japan stock prediction value.csv


In [4]:
# Dynamically handle dataset shape
num_rows, num_cols = df.shape
if num_rows < 2 or num_cols < 2:
    raise ValueError(f"Dataset must have at least 2 rows and 2 columns, got {df.shape}")

# Ensure timestamp column contains sequential integers starting from 1
if not df.iloc[:, 0].equals(pd.Series(range(1, num_rows + 1))):
    raise ValueError("Timestamp column must contain sequential integers starting from 1")

# Rename first column to 'timestamp' for consistency
df.columns = ['timestamp'] + list(df.columns[1:])

# Step 2: Analyze each price column (all columns except timestamp)
price_columns = df.columns[1:]  # All columns except timestamp
results = []

for col in price_columns:
    prices = df[col]
    max_price = prices.max()
    min_price = prices.min()
    max_idx = prices.idxmax()
    min_idx = prices.idxmin()
    max_time = df['timestamp'].iloc[max_idx]
    min_time = df['timestamp'].iloc[min_idx]
    std_dev = prices.std()

    results.append({
        'asset': col,
        'max_price': max_price,
        'min_price': min_price,
        'max_time': max_time,
        'min_time': min_time,
        'std_dev': std_dev
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Step 3: Identify buy and sell portfolios
sell_portfolio = results_df[results_df['max_time'] < results_df['min_time']]['asset'].tolist()
buy_portfolio = results_df[results_df['max_time'] > results_df['min_time']]['asset'].tolist()

# Step 4: Calculate weights using inverse of standard deviations
results_df['inv_std'] = 1 / results_df['std_dev'].replace(0, np.inf)
total_inv_std = results_df['inv_std'].sum()
results_df['weight'] = results_df['inv_std'] / total_inv_std

# Step 5: Allocate total investment ($1,000)
total_investment = 1000

# Calculate total weight for buy and sell portfolios
buy_weights_sum = results_df[results_df['asset'].isin(buy_portfolio)]['weight'].sum()
sell_weights_sum = results_df[results_df['asset'].isin(sell_portfolio)]['weight'].sum()

# Normalize weights to ensure full allocation
total_weight = buy_weights_sum + sell_weights_sum
if total_weight > 0:
    buy_allocation = total_investment * (buy_weights_sum / total_weight)
    sell_allocation = total_investment * (sell_weights_sum / total_weight)
else:
    buy_allocation = 0
    sell_allocation = 0

# Step 6: Reallocate weights within each portfolio
buy_weights = results_df[results_df['asset'].isin(buy_portfolio)][['asset', 'weight']].copy()
sell_weights = results_df[results_df['asset'].isin(sell_portfolio)][['asset', 'weight']].copy()

if buy_weights_sum > 0:
    buy_weights['portfolio_weight'] = buy_weights['weight'] / buy_weights_sum
else:
    buy_weights['portfolio_weight'] = 0

if sell_weights_sum > 0:
    sell_weights['portfolio_weight'] = sell_weights['weight'] / sell_weights_sum
else:
    sell_weights['portfolio_weight'] = 0

# Step 7: Calculate integer shares and balance
portfolio = []
balance = total_investment

# Buy portfolio: Buy at min price, sell at max price
for _, row in buy_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        allocation = buy_allocation * row['portfolio_weight']
        shares = int(allocation / min_price)  # Integer shares
        if shares > 0:  # Only include if shares are allocated
            cost = shares * min_price
            balance -= cost
            portfolio.append({
                'asset': asset,
                'type': 'buy',
                'shares': shares,
                'buy_price': min_price,
                'sell_price': max_price,
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost,
                'revenue': shares * (max_price - min_price)  # Profit per share
            })

# Sell portfolio: Sell short at max price, cover at min price
for _, row in sell_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        allocation = sell_allocation * row['portfolio_weight']
        shares = int(allocation / max_price)  # Integer shares
        if shares > 0:
            cost = shares * max_price
            balance -= cost
            portfolio.append({
                'asset': asset,
                'type': 'sell',
                'shares': shares,
                'buy_price': min_price,  # Cover price
                'sell_price': max_price,  # Short price
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost,
                'revenue': shares * (max_price - min_price)  # Profit from short
            })

portfolio_df = pd.DataFrame(portfolio)

In [5]:
# Step 8: Calculate portfolio metrics
risk_free_rate = 0.01527 # Annual risk-free rate

# Function to calculate portfolio metrics
def calculate_portfolio_metrics(portfolio_df, portfolio_type, df_prices):
    if portfolio_df.empty:
        return {'return': 0, 'volatility': 0, 'sharpe_ratio': 0}

    if portfolio_type == 'buy':
        df = portfolio_df[portfolio_df['type'] == 'buy']
    else:
        df = portfolio_df[portfolio_df['type'] == 'sell']

    if df.empty:
        return {'return': 0, 'volatility': 0, 'sharpe_ratio': 0}

    total_cost = df['cost'].sum()
    if total_cost == 0:
         return {'return': 0, 'volatility': 0, 'sharpe_ratio': 0}

    # Calculate daily portfolio returns
    daily_returns = []
    # The number of trading days is the number of rows in the price data minus 1
    num_trading_days = df_prices.shape[0] - 1

    for i in range(num_trading_days):
        current_day_prices = df_prices.iloc[i, 1:] # Exclude timestamp
        next_day_prices = df_prices.iloc[i+1, 1:] # Exclude timestamp
        daily_portfolio_return = 0

        for _, row in df.iterrows():
            asset = row['asset']
            shares = row['shares']
            buy_price = row['buy_price'] # This is the price at the time of the trade

            if asset in current_day_prices.index and asset in next_day_prices.index:
                # Find the price of the asset on the current day and the next day
                price_today = current_day_prices[asset]
                price_tomorrow = next_day_prices[asset]

                if row['type'] == 'buy':
                    # For buy trades, return is based on price increase
                    asset_return = (price_tomorrow - price_today) / price_today if price_today != 0 else 0
                else:
                    # For sell trades (shorting), return is based on price decrease
                    asset_return = (price_today - price_tomorrow) / price_today if price_today != 0 else 0

                # Weight the return by the proportion of this asset's cost in the portfolio's total cost
                asset_cost_proportion = (shares * buy_price) / total_cost
                daily_portfolio_return += asset_return * asset_cost_proportion

        daily_returns.append(daily_portfolio_return)

    daily_avg_return = np.mean(daily_returns) if daily_returns else 0
    # Annualize return for Sharpe ratio (assuming 252 trading days in a year)
    annualized_return = (1 + daily_avg_return) ** 245 - 1

    # Volatility: std of daily returns, annualized
    volatility = np.std(daily_returns) * np.sqrt(252) if daily_returns else 0
    sharpe_ratio = (annualized_return - risk_free_rate) / volatility if volatility > 0 else 0

    return {
        'return': daily_avg_return, # Keep as daily average return
        'volatility': volatility,
        'sharpe_ratio': sharpe_ratio
    }

# Modify step 7 to allow for fractional shares and ensure portfolio is not empty
portfolio = []
balance = total_investment

# Buy portfolio: Buy at min price, sell at max price
for _, row in buy_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        allocation = buy_allocation * row['portfolio_weight']
        shares = allocation / min_price  # Allow fractional shares
        if shares > 0:
            cost = shares * min_price
            # balance -= cost # Don't subtract cost here, it's part of investment_made
            portfolio.append({
                'asset': asset,
                'type': 'buy',
                'shares': shares,
                'buy_price': min_price,
                'sell_price': max_price,
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost,
                'revenue': shares * (max_price - min_price)  # Profit per share
            })

# Sell portfolio: Sell short at max price, cover at min price
for _, row in sell_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        allocation = sell_allocation * row['portfolio_weight']
        shares = allocation / max_price  # Allow fractional shares
        if shares > 0:
            cost = shares * max_price
            # balance -= cost # Don't subtract cost here, it's part of investment_made
            portfolio.append({
                'asset': asset,
                'type': 'sell',
                'shares': shares,
                'buy_price': min_price,  # Cover price
                'sell_price': max_price,  # Short price
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost,
                'revenue': shares * (max_price - min_price)  # Profit from short
            })

portfolio_df = pd.DataFrame(portfolio)

# Ensure portfolio_df is not empty before calculating metrics
if not portfolio_df.empty:
    buy_metrics = calculate_portfolio_metrics(portfolio_df, 'buy', df)
    sell_metrics = calculate_portfolio_metrics(portfolio_df, 'sell', df)

    # Combined portfolio metrics
    total_cost = portfolio_df['cost'].sum()
    total_revenue = portfolio_df['revenue'].sum()
    investment_made = total_cost  # Investment made is the total cost of trades
    portfolio_final_balance = investment_made + total_revenue
    overall_return = total_revenue / investment_made if investment_made > 0 else 0
    annualized_overall_return = (1 + overall_return) ** 252 - 1 # Assuming 252 trading days

    # Calculate overall volatility and Sharpe ratio based on combined daily returns
    overall_daily_returns = []
    num_trading_days = df.shape[0] - 1
    total_investment_allocated = buy_allocation + sell_allocation

    for i in range(num_trading_days):
        current_day_prices = df.iloc[i, 1:] # Exclude timestamp
        next_day_prices = df.iloc[i+1, 1:] # Exclude timestamp
        daily_overall_return = 0

        for _, row in portfolio_df.iterrows():
            asset = row['asset']
            shares = row['shares']
            buy_price = row['buy_price'] # This is the price at the time of the trade

            if asset in current_day_prices.index and asset in next_day_prices.index:
                price_today = current_day_prices[asset]
                price_tomorrow = next_day_prices[asset]

                if row['type'] == 'buy':
                    asset_return = (price_tomorrow - price_today) / price_today if price_today != 0 else 0
                else:
                    asset_return = (price_today - price_tomorrow) / price_today if price_today != 0 else 0

                # Weight the return by the proportion of this asset's initial allocation in the total allocated investment
                asset_allocation_proportion = (shares * buy_price) / total_investment_allocated if total_investment_allocated > 0 else 0
                daily_overall_return += asset_return * asset_allocation_proportion

        overall_daily_returns.append(daily_overall_return)

    overall_volatility = np.std(overall_daily_returns) * np.sqrt(252) if overall_daily_returns else 0
    overall_sharpe = (annualized_overall_return - risk_free_rate) / overall_volatility if overall_volatility > 0 else 0

    # Daily and yearly ROI
    daily_roi = overall_return # Overall return calculated earlier is total return, not daily
    yearly_roi = annualized_overall_return # Use the annualized overall return

    # Step 9: Output results
    print("Buy Portfolio:")
    print(buy_weights[['asset', 'portfolio_weight']])
    print(f"Buy Allocation: ${buy_allocation:.2f}")
    print(f"Buy Return: {buy_metrics['return']*100:.4f}%")
    print(f"Buy Sharpe Ratio: {buy_metrics['sharpe_ratio']:.4f}")
    print("\nSell Portfolio:")
    print(sell_weights[['asset', 'portfolio_weight']])
    print(f"Sell Allocation: ${sell_allocation:.2f}")
    print(f"Sell Return: {sell_metrics['return']*100:.4f}%")
    print(f"Sell Sharpe Ratio: {sell_metrics['sharpe_ratio']:.4f}")
    print("\nPortfolio Trades:")
    print(portfolio_df[['asset', 'type', 'shares', 'buy_price', 'sell_price', 'buy_time', 'sell_time', 'cost', 'revenue']])
    print(f"\nInvestment Made: ${investment_made:.2f}")
    print(f"Remaining Balance: ${balance:.2f}")
    print(f"Portfolio Final Balance: ${portfolio_final_balance:.2f}")
    print(f"Overall Return: {overall_return*100:.4f}%")
    print(f"Daily ROI: {daily_roi*100:.4f}%") # This is actually total return, not daily
    print(f"Yearly ROI: {yearly_roi*100:.4f}%")
    print(f"Overall Portfolio Risk (Volatility): {overall_volatility:.4f}")
    print(f"Yearly Sharpe Ratio: {overall_sharpe:.4f}")

    # Save portfolio details to a file
    portfolio_df.to_csv('portfolio_output.csv', index=False)
else:
    print("No trades were made for the specified portfolios.")
    # Create an empty portfolio_output.csv file if no trades were made
    pd.DataFrame(columns=['asset', 'type', 'shares', 'buy_price', 'sell_price', 'buy_time', 'sell_time', 'cost', 'revenue']).to_csv('portfolio_output.csv', index=False)

Buy Portfolio:
     asset  portfolio_weight
1   4043.T          0.062236
2   4324.T          0.079790
4   5332.T          0.077307
5   5411.T          0.079319
6   5631.T          0.024174
8   6724.T          0.056641
9   6762.T          0.108375
10  6841.T          0.027125
11  7011.T          0.016350
12  7267.T          0.129271
13  8031.T          0.038259
14  8058.T          0.076859
15  8253.T          0.029649
16  9021.T          0.075452
17  7203.T          0.040870
18  7272.T          0.078322
Buy Allocation: $870.37
Buy Return: 0.0033%
Buy Sharpe Ratio: -3.3687

Sell Portfolio:
    asset  portfolio_weight
0  4543.T          0.303833
3  4704.T          0.086462
7  6702.T          0.609705
Sell Allocation: $129.63
Sell Return: 0.0017%
Sell Sharpe Ratio: -2.6802

Portfolio Trades:
     asset  type    shares     buy_price    sell_price  buy_time  sell_time  \
0   4043.T   buy  0.019349   2799.511000   2815.622000        23        127   
1   4324.T   buy  0.022322   3111.124800   

In [6]:
# prompt: provide the result as csv file as output in computer

files.download('portfolio_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>